# Train a Tiny Neural Classifier

Complete a small PyTorch training loop, return the trained model, and let Colab validate its interface, predictions and accuracy.

This is a formative Colab practical. Work through the three numbered stages below. Every validation attempt shows pass/fail here and synchronizes its latest result to the learning platform.

## Step 1 — connect this notebook

Choose the Colab hardware accelerator before running this cell. Changing CPU/GPU or reconnecting to a fresh runtime erases the connection and requires **Start a new Colab session** on the lesson page. Then run this protected setup cell once.

In [ ]:
# Step 1 — run this setup cell once before editing your solution.
from getpass import getpass
import base64
import hashlib
import importlib.util
from importlib.metadata import PackageNotFoundError, version as package_version
import json
from packaging.version import Version
import re
import requests
import shutil
import subprocess
from urllib.parse import urlsplit

EXPECTED_ACTIVITY_ID = "train-tiny-neural-classifier"

def northstar_api_data(response):
    try:
        payload = response.json()
    except ValueError as error:
        raise RuntimeError(f"Platform returned a non-JSON response ({response.status_code})") from error
    if not response.ok or payload.get("ok") is not True:
        message = payload.get("error", {}).get("message", response.text)
        raise RuntimeError(f"Platform request failed ({response.status_code}): {message}")
    return payload["data"]

def decode_connection_code(value):
    value = value.strip()
    if not value.startswith("NS1."):
        raise RuntimeError("This is not a Northstar Colab connection code.")
    encoded = value[4:]
    if not re.fullmatch(r"[A-Za-z0-9_-]+", encoded):
        raise RuntimeError("The Colab connection code is malformed.")
    try:
        padding = "=" * ((4 - len(encoded) % 4) % 4)
        details = json.loads(base64.urlsafe_b64decode(encoded + padding).decode("utf-8"))
    except (ValueError, UnicodeDecodeError, json.JSONDecodeError) as error:
        raise RuntimeError("The Colab connection code could not be decoded.") from error
    api_base = details.get("apiBase") if isinstance(details, dict) else None
    pairing_code = details.get("pairingCode") if isinstance(details, dict) else None
    if not isinstance(api_base, str) or not isinstance(pairing_code, str):
        raise RuntimeError("The Colab connection code is incomplete.")
    parsed_api = urlsplit(api_base)
    if (parsed_api.scheme != "https" or not parsed_api.hostname or parsed_api.username or
            parsed_api.password or parsed_api.query or parsed_api.fragment or parsed_api.path not in ("", "/")):
        raise RuntimeError("The connection code does not contain a valid public HTTPS API origin.")
    if not re.fullmatch(r"[A-Z0-9-]{6,32}", pairing_code):
        raise RuntimeError("The connection code contains an invalid pairing secret.")
    return api_base.rstrip("/"), pairing_code

def receive_connection_from_platform(timeout_ms=8000):
    try:
        from google.colab import output as colab_output
    except ImportError:
        return None
    bridge_script = f"""
new Promise((resolve) => {{
  const requestId = crypto.randomUUID();
  let finished = false;
  const finish = (value) => {{
    if (finished) return;
    finished = true;
    clearTimeout(timer);
    window.removeEventListener('message', receive);
    resolve(value);
  }};
  const receive = (event) => {{
    const message = event.data;
    if (!message || message.type !== 'northstar-colab-connection' ||
        message.protocol !== 'NS1' || message.requestId !== requestId ||
        typeof message.connectionCode !== 'string' || !message.connectionCode.startsWith('NS1.')) return;
    try {{ window.top.opener = null; }} catch (error) {{}}
    finish(message.connectionCode);
  }};
  window.addEventListener('message', receive);
  const timer = setTimeout(() => finish(null), {timeout_ms});
  try {{
    const platform = window.top.opener;
    if (!platform) return finish(null);
    platform.postMessage({{
      type: 'northstar-colab-ready',
      protocol: 'NS1',
      requestId,
    }}, '*');
  }} catch (error) {{
    finish(null);
  }}
}})
"""
    try:
        connection_code = colab_output.eval_js(bridge_script)
    except Exception:
        return None
    return connection_code if isinstance(connection_code, str) else None

def colab_environment_metrics():
    if not shutil.which("nvidia-smi"):
        return {"colabGpuAvailable": False}
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    line = query.stdout.strip().splitlines()[0] if query.stdout.strip() else ""
    if query.returncode != 0 or not line:
        return {"colabGpuAvailable": False}
    name, memory_mib = [part.strip() for part in line.split(",", 1)]
    return {"colabGpuAvailable": True, "colabGpuName": name, "gpuMemoryMiB": int(memory_mib)}

connection_cache = globals().get("__northstar_colab_only_connection__")
cached_code = connection_cache.get("connectionCode") if isinstance(connection_cache, dict) else None
CONNECTION_CODE = receive_connection_from_platform(1500 if cached_code else 8000)
if CONNECTION_CODE:
    print("Connection received automatically from the learning platform.")
elif isinstance(cached_code, str):
    CONNECTION_CODE = cached_code
    print("Reusing the connected Colab practical.")
else:
    CONNECTION_CODE = getpass("Automatic connection unavailable. Paste a NEW connection code from the lesson: ").strip()
connection_digest = hashlib.sha256(CONNECTION_CODE.encode("utf-8")).hexdigest()

if isinstance(connection_cache, dict) and connection_cache.get("digest") == connection_digest:
    API_BASE = connection_cache["apiBase"]
    RUN_ID = connection_cache["runId"]
    AUTH = connection_cache["auth"]
    bundle = connection_cache["bundle"]
else:
    API_BASE, PAIRING_CODE = decode_connection_code(CONNECTION_CODE)
    claim_response = requests.post(
        f"{API_BASE}/api/colab/claim",
        json={"pairingCode": PAIRING_CODE},
        timeout=30,
    )
    if claim_response.status_code in (404, 409, 410):
        raise RuntimeError(
            "This connection code is expired or was already used. Return to the lesson page, "
            "choose 'Start a new Colab session', and use its new connection. Runtime restarts "
            "and accelerator changes always require a new platform session."
        )
    claim = northstar_api_data(claim_response)
    RUN_ID = claim["runId"]
    AUTH = {"Authorization": f"Bearer {claim['accessToken']}"}
    bundle = northstar_api_data(requests.get(
        f"{API_BASE}/api/colab/runtime/{RUN_ID}/bundle",
        headers=AUTH,
        timeout=30,
    ))
    __northstar_colab_only_connection__ = {
        "digest": connection_digest,
        "connectionCode": CONNECTION_CODE,
        "apiBase": API_BASE,
        "runId": RUN_ID,
        "auth": AUTH,
        "bundle": bundle,
    }
del CONNECTION_CODE

if bundle.get("activityId") != EXPECTED_ACTIVITY_ID:
    raise RuntimeError("This notebook does not match the activity opened on the learning platform.")
execution = bundle["execution"]
if execution.get("workspace") != "colab_only":
    raise RuntimeError("This activity is not configured for a Colab-only workspace.")

runtime = execution["runtime"]
problems = []
for dependency in runtime["dependencies"]:
    import_root = dependency["importName"].split(".", 1)[0]
    if importlib.util.find_spec(import_root) is None:
        problems.append(f"missing {dependency['package']}")
        continue
    minimum = dependency.get("minimumVersion")
    if minimum:
        try:
            installed = package_version(dependency["package"])
            if Version(installed) < Version(minimum):
                problems.append(f"{dependency['package']} {installed} (needs >= {minimum})")
        except PackageNotFoundError:
            problems.append(f"missing {dependency['package']}")
if problems:
    raise RuntimeError("Colab runtime requirements are not satisfied: " + "; ".join(problems))

__northstar_environment_metrics__ = colab_environment_metrics()
__northstar_runtime_error__ = None
if runtime["accelerator"] == "gpu_required" and not __northstar_environment_metrics__.get("colabGpuAvailable"):
    __northstar_runtime_error__ = "This activity requires an NVIDIA GPU. Choose Runtime > Change runtime type in Colab."
minimum_gpu_memory = runtime.get("minimumGpuMemoryMb")
if minimum_gpu_memory and __northstar_environment_metrics__.get("gpuMemoryMiB", 0) < minimum_gpu_memory:
    __northstar_runtime_error__ = f"This activity requires at least {minimum_gpu_memory} MiB of GPU memory."

print(f"Connected to: {bundle['activityId']}")
print(f"Requested accelerator: {runtime['accelerator']}")
print(f"GPU allocated: {__northstar_environment_metrics__.get('colabGpuAvailable', False)}")
if runtime["accelerator"] == "gpu_preferred" and not __northstar_environment_metrics__.get("colabGpuAvailable", False):
    print("CPU fallback is active. If you want a GPU, select it before Step 1; changing runtime later requires a new platform session.")
print("Setup complete. Edit and run Step 2, then run Step 3 to validate.")

## Step 2 — edit and run your solution

Complete the four training operations inside the loop: optimizer.zero_grad(), calculate logits and cross-entropy loss, loss.backward(), then optimizer.step(). Return the trained model. Colab supplies deterministic two-feature training data and checks the returned PyTorch module on separate validation rows.

This is your learner-owned code cell. Edit it freely and run it before each validation attempt.

In [ ]:
import torch
from torch import nn

def train_model(train_x, train_y):
    """Train and return a tiny two-class PyTorch model."""
    torch.manual_seed(17)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    features = train_x.to(device)
    labels = train_y.to(device)

    model = nn.Sequential(
        nn.Linear(2, 8),
        nn.ReLU(),
        nn.Linear(8, 2),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    loss_function = nn.CrossEntropyLoss()

    for _ in range(100):
        # TODO: clear old gradients, calculate logits and loss,
        # backpropagate, and update the parameters.
        pass

    return model

## Step 3 — validate and synchronize

Run this protected validation cell after every change. You can submit multiple attempts from this same notebook session.

In [ ]:
# Step 3 — run this cell after every solution change. It can submit multiple attempts.
import contextlib
import io
import json
import time
import traceback
import uuid
from IPython.display import Markdown, display

required_setup = ["API_BASE", "RUN_ID", "AUTH", "bundle", "__northstar_environment_metrics__"]
missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError("Run Step 1 before validating your solution.")

def northstar_json_safe(value):
    try:
        return json.loads(json.dumps(value))
    except (TypeError, ValueError):
        return {"repr": repr(value)}

captured_stdout = io.StringIO()
captured_stderr = io.StringIO()
namespace = dict(globals())
namespace["__northstar_tests__"] = bundle["validation"]["tests"]
execution_error = globals().get("__northstar_runtime_error__")
started_at = time.perf_counter()
with contextlib.redirect_stdout(captured_stdout), contextlib.redirect_stderr(captured_stderr):
    if execution_error is None:
        try:
            exec(compile(bundle["validation"]["harnessCode"], "northstar_validation.py", "exec"), namespace)
        except (Exception, SystemExit):
            execution_error = traceback.format_exc()
execution_seconds = time.perf_counter() - started_at
if execution_error is None and execution_seconds > bundle["execution"]["runtime"]["maxExecutionSeconds"]:
    execution_error = f"Execution exceeded the {bundle['execution']['runtime']['maxExecutionSeconds']} second activity limit."

validation = namespace.get("__northstar_validation__")
if execution_error is not None:
    validation = {"status": "error", "tests": [], "metrics": {}}
elif not isinstance(validation, dict):
    execution_error = "Validation did not produce a structured result."
    validation = {"status": "error", "tests": [], "metrics": {}}

metrics = validation.get("metrics", {})
if not isinstance(metrics, dict):
    metrics = {"reportedMetrics": northstar_json_safe(metrics)}
metrics.update(__northstar_environment_metrics__)
metrics["executionSeconds"] = round(execution_seconds, 4)
stdout = captured_stdout.getvalue()
stderr = captured_stderr.getvalue() + (execution_error or "")
callback = {
    "executionId": str(uuid.uuid4()),
    "status": validation.get("status", "error"),
    "tests": northstar_json_safe(validation.get("tests", [])),
    "returnValue": northstar_json_safe(validation.get("returnValue")),
    "metrics": northstar_json_safe(metrics),
    "stdout": stdout[:50000],
    "stderr": stderr[:50000],
    "outputTruncated": len(stdout) > 50000 or len(stderr) > 50000,
}

pending_delivery = globals().get("__northstar_colab_only_pending_delivery__")
if isinstance(pending_delivery, dict) and pending_delivery.get("runId") == RUN_ID:
    callback = pending_delivery["callback"]
    print("Retrying delivery of the previously prepared validation attempt.")
else:
    __northstar_colab_only_pending_delivery__ = {"runId": RUN_ID, "callback": callback}

accepted = None
delivery_error = None
for delay_seconds in (0, 1, 2):
    if delay_seconds:
        time.sleep(delay_seconds)
    try:
        accepted = northstar_api_data(requests.post(
            f"{API_BASE}/api/colab/runtime/{RUN_ID}/result",
            headers={**AUTH, "Content-Type": "application/json"},
            json=callback,
            timeout=30,
        ))
        break
    except (requests.RequestException, RuntimeError) as error:
        delivery_error = error
if accepted is None:
    raise RuntimeError("Validation finished, but the platform callback failed after three retries.") from delivery_error
__northstar_colab_only_pending_delivery__ = None

status = accepted["result"]["status"]
if status == "passed":
    display(Markdown("## ✅ Passed in Colab"))
elif status == "failed":
    display(Markdown("## ❌ Not passed yet — revise your solution and try again"))
else:
    display(Markdown("## ⚠️ Validation error"))
print(f"Attempt {accepted['attemptNumber']} synchronized to the learning platform.")
for result in accepted["result"]["tests"]:
    marker = "PASS" if result["passed"] else "FAIL"
    print(f"[{marker}] {result['id']}: {result.get('message', '')}")
print("\nMetrics")
print(json.dumps(accepted["result"]["metrics"], indent=2))
if accepted["result"].get("returnValue") is not None:
    print("\nReturn value")
    print(json.dumps(accepted["result"]["returnValue"], indent=2))
if accepted["result"].get("stderr"):
    print("\nErrors")
    print(accepted["result"]["stderr"])
print("\nEdit and rerun Step 2, then rerun this cell for another attempt.")

When your latest attempt passes, return to the lesson page. The result should already be visible there.